In [6]:
BRANCH = "test-ramcharan"
REPO_URL = "git@github.com:St1p42/sglang.git"
REPO_DIR = "/content/sglang"

In [7]:
%%bash
set -e
mkdir -p /root/.ssh
chmod 700 /root/.ssh
if [ ! -f /root/.ssh/id_ed25519 ]; then
  ssh-keygen -t ed25519 -C "colab" -f /root/.ssh/id_ed25519 -N ""
fi
ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
chmod 600 /root/.ssh/known_hosts
echo "Public key:"
cat /root/.ssh/id_ed25519.pub

Generating public/private ed25519 key pair.
Your identification has been saved in /root/.ssh/id_ed25519
Your public key has been saved in /root/.ssh/id_ed25519.pub
The key fingerprint is:
SHA256:CcBXyoK+i8kCuolLGutQ9WzNJ+3OPbl5Z/LoAgu24Qw colab
The key's randomart image is:
+--[ED25519 256]--+
|   ..  ..        |
|   ..o..         |
|  . o.+          |
| . . + + o       |
|  o   + S o      |
|.. . .E ++.      |
|=..    = +.o .   |
|O* .    +o..+.o.o|
|%=.       o +*o=.|
+----[SHA256]-----+
Public key:
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAINYr9WasftcgxfYykOjWF9c3FD35JIUb+rccG1nLgNTz colab


In [9]:
%%bash
set -e

ssh -T git@github.com || true

Hi rjagalamari! You've successfully authenticated, but GitHub does not provide shell access.


In [10]:
import os
os.environ["BRANCH"] = BRANCH
os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = REPO_DIR

In [11]:
%%bash
set -e
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
fi
cd "$REPO_DIR"
git fetch origin
git checkout "$BRANCH"
git pull origin "$BRANCH"
git branch --show-current

git log --oneline -1

Branch 'test-ramcharan' set up to track remote branch 'test-ramcharan' from 'origin'.
Already up to date.
test-ramcharan
cabc91eaf Add custom HiCache routing and benchmark notebook


Cloning into '/content/sglang'...
From github.com:St1p42/sglang
   d04d87802..1402e942f  test-khoa  -> origin/test-khoa
Switched to a new branch 'test-ramcharan'
From github.com:St1p42/sglang
 * branch                test-ramcharan -> FETCH_HEAD


In [12]:
%%bash
echo "=== GPU Info ==="
nvidia-smi || echo "No GPU attached or nvidia-smi failed"
echo -e "\n=== CPU Info ==="
lscpu | grep 'Model name\|Socket(s)\|Core(s) per socket\|Thread(s) per core'
echo -e "\n=== RAM Info ==="
free -h

=== GPU Info ===
Wed Apr 29 20:27:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   54C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

In [13]:
%%bash
set -e
cd /content/sglang
python -m pip uninstall -y sglang sgl-kernel flashinfer-python || true
python -m pip install -e python

Obtaining file:///content/sglang/python
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 9.4 MB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.


In [14]:
# You should see something like:
# torch: 2.9.1+cu128
# sglang version: 0.5.6
# sglang path: ['/content/sglang/python/sglang']
# sglang spec origin: /content/sglang/python/sglang/__init__.py
# sgl_kernel: 0.3.18.post2
# cuda: True NVIDIA L4
import sys

# Make sure Python imports the actual package in your repo
sys.path = [p for p in sys.path if p != "/content/sglang"]
sys.path.insert(0, "/content/sglang/python")

import sglang, torch, sgl_kernel
from importlib.metadata import version

print("torch:", torch.__version__)
print("sglang version:", version("sglang"))
print("sglang path:", list(sglang.__path__))
print("sglang spec origin:", sglang.__spec__.origin)
print("sgl_kernel:", getattr(sgl_kernel, "__version__", "ok"))
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

torch: 2.9.1+cu128
sglang version: 0.5.6
sglang path: ['/content/sglang/python/sglang']
sglang spec origin: /content/sglang/python/sglang/__init__.py
sgl_kernel: 0.3.18.post2
cuda: True NVIDIA L4


In [15]:
%cd /content/sglang
!git branch --show-current
!git status --short --branch

/content/sglang
test-ramcharan
## test-ramcharan...origin/test-ramcharan


In [16]:
!pip install -U nvidia-cudnn-cu12==9.16.0.29

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.7/647.7 MB 2.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cudnn-cu12
    Found existing installation: nvidia-cudnn-cu12 9.10.2.21
    Uninstalling nvidia-cudnn-cu12-9.10.2.21:
      Successfully uninstalled nvidia-cudnn-cu12-9.10.2.21
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.9.1 requires nvidia-cudnn-cu12==9.10.2.21; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.16.0.29 which is incompatible.


In [39]:
%cd /content/sglang
!nohup python3 -m sglang.launch_server \
      --model-path Qwen/Qwen2.5-1.5B-Instruct \
      --port 30000 \
      --mem-fraction-static 0.65 \
      --enable-hierarchical-cache \
      --hicache-impl vanilla \
      --hicache-size 12 \
      > server.log 2>&1 &

/content/sglang


In [47]:
!tail -100 server.log
!curl http://127.0.0.1:30000/get_model_info

[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla radix attention implementation.
we are in else vanilla radix attention
[2026-04-29 20:49:42] Using vanilla r

In [48]:
!curl http://127.0.0.1:30000/get_model_info

{"model_path":"Qwen/Qwen2.5-1.5B-Instruct","tokenizer_path":"Qwen/Qwen2.5-1.5B-Instruct","is_generation":true,"preferred_sampling_params":null,"weight_version":"default","has_image_understanding":false,"has_audio_understanding":false}

In [49]:
!pip install av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 74.5 MB/s eta 0:00:00


In [38]:
!pkill -f launch_server

In [53]:

  !python3 benchmark/hicache/bench_multiturn.py \
      --model-path Qwen/Qwen2.5-1.5B-Instruct \
      --port 30000 \
      --disable-auto-run \
      --request-rate 10 \
      --seed 1 \
      --num-clients 64 \
      --max-parallel 32 \
      --num-rounds 3


2026-04-29 20:58:34.148544: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-29 20:58:34.218359: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Running with specified request rate...
/tmp/ShareGPT_V3_unfiltered_cleaned_split.json: 100% 642M/642M [00:05<00:00, 115MB/s]
#Input tokens: 32768
#Output tokens: 4096
#Input tokens: 65536
#Output tokens: 8192
100% 192/192 [00:41<00:00,  4.64it/s]
All requests completed
Performance metrics summary:
  Total requests: 192 at 10.0 requests per second
  Average Prompt Length: 